# Week 64, Day 4: Learning kernels from data: the marginal likelihood

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w064/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The two terms across length-scales

For length-scales from 0.05 to 10 (log-spaced), compute the data-fit term, the complexity term and their sum (plus the constant) for 40 noisy samples of a function drawn from a GP with length-scale 0.7.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import cho_factor, cho_solve

def k_se(A, B, ell, sf2=1.0):
    return sf2 * np.exp(-0.5 * ((A[:, None] - B[None, :]) / ell) ** 2)

g0 = np.random.default_rng(644)
X = np.sort(g0.uniform(-3, 3, 40))
f_true = np.linalg.cholesky(k_se(X, X, 0.7) + 1e-8 * np.eye(40)) @ g0.normal(size=40)
y = f_true + 0.3 * g0.normal(size=40)

def lml_terms(ell, sf2, sn2):
    C = k_se(X, X, ell, sf2) + sn2 * np.eye(len(X))
    c, low = cho_factor(C, lower=True)
    fit = ...
    comp = -np.sum(np.log(np.diag(c)))
    return fit, comp, fit + comp - 0.5 * len(X) * np.log(2 * np.pi)

ells = np.exp(np.linspace(np.log(0.05), np.log(10), 40))
fit_t, comp_t, lml_t = map(np.array, zip(*[lml_terms(l, 1.0, 0.09) for l in ells]))
print(f"best length-scale {ells[np.argmax(lml_t)]:.2f}")
plt.semilogx(ells, fit_t, label="data fit"); plt.semilogx(ells, comp_t, label="complexity"); plt.semilogx(ells, lml_t, "k", label="log marginal likelihood")
plt.legend(); plt.xlabel("length-scale"); plt.show()

In [ ]:
import numpy as np
def test_terms():
    "the data-fit term improves and the complexity term worsens as the length-scale shrinks"
    assert fit_t[0] > fit_t[-1] and comp_t[0] < comp_t[-1]
def test_peak():
    "the log marginal likelihood peaks at an intermediate length-scale, within a factor of 2 of the generating one (0.7)"
    best = ells[np.argmax(lml_t)]
    assert ells[0] < best < ells[-1] and 0.35 < best < 1.4

test_terms()
test_peak()
print("All checks passed")

### Exercise: Type II maximum likelihood

Write neg_lml(log_params) returning minus the log marginal likelihood for (log ell, log sf2, log sn2), and minimise it with scipy.optimize.minimize (method L-BFGS-B) from several starting points, keeping the best.

In [ ]:
from scipy.optimize import minimize

def neg_lml(lp):
    ell, sf2, sn2 = np.exp(lp)
    return ...

best = None
for start in ([0.0, 0.0, -1.0], [-2.0, 1.0, -3.0], [1.5, -1.0, 0.0]):
    res = minimize(neg_lml, np.array(start), method="L-BFGS-B")
    if best is None or res.fun < best.fun:
        best = res
ell_hat, sf2_hat, sn2_hat = np.exp(best.x)
print(f"length-scale {ell_hat:.3f} (true 0.7), signal sd {np.sqrt(sf2_hat):.3f}, noise sd {np.sqrt(sn2_hat):.3f} (true 0.3); log marginal likelihood {-best.fun:.2f}")

In [ ]:
import numpy as np
def test_recover():
    "the fitted noise sd is within 30% of 0.3 and the length-scale within a factor of 1.6 of 0.7"
    assert abs(np.sqrt(sn2_hat) / 0.3 - 1) < 0.3 and 0.7 / 1.6 < ell_hat < 0.7 * 1.6

test_recover()
print("All checks passed")

### Exercise: Squared exponential against periodic

Simulate a firing-rate rhythm (period 1.3 s) observed for 4 s with noise, fit both kernels by maximising the marginal likelihood (the periodic kernel also learns its period), and compare their best marginal likelihoods and their errors when predicting 4 to 6 s.

In [ ]:
gc = np.random.default_rng(6441)
tt = np.sort(gc.uniform(0, 4, 50))
rhythm = lambda t: np.sin(2 * np.pi * t / 1.3) + 0.5 * np.sin(4 * np.pi * t / 1.3 + 0.7)
yr = rhythm(tt) + 0.25 * gc.normal(size=50)
t_out = np.linspace(4, 6, 50)

kern = {
    "se": lambda A, B, p: np.exp(p[1]) * np.exp(-0.5 * ((A[:, None] - B[None, :]) / np.exp(p[0])) ** 2),
    "periodic": lambda A, B, p: np.exp(p[1]) * np.exp(-2 * np.sin(np.pi * np.abs(A[:, None] - B[None, :]) / np.exp(p[3])) ** 2 / np.exp(p[0]) ** 2),
}
def nlml(p, name):
    C = kern[name](tt, tt, p) + np.exp(p[2]) * np.eye(len(tt))
    c, low = cho_factor(C, lower=True)
    return 0.5 * yr @ cho_solve((c, low), yr) + np.sum(np.log(np.diag(c))) + 0.5 * len(tt) * np.log(2 * np.pi)

best_lml, err_out = {}, {}
starts = {"se": [[0.0, 0.0, -2.0], [-1.0, 0.0, -2.0]], "periodic": [[0.0, 0.0, -2.0, np.log(p0)] for p0 in (1.0, 1.3, 1.6)]}
for name in kern:
    res = min((minimize(nlml, np.array(s0), args=(name,), method="L-BFGS-B") for s0 in starts[name]), key=lambda r: r.fun)
    best_lml[name] = -res.fun
    p = res.x
    C = kern[name](tt, tt, p) + np.exp(p[2]) * np.eye(len(tt))
    pred = ...                                          # posterior mean at t_out
    err_out[name] = np.sqrt(np.mean((pred - rhythm(t_out)) ** 2))
    extra = f", period {np.exp(p[3]):.3f} s" if name == "periodic" else ""
    print(f"{name:9s} log marginal likelihood {best_lml[name]:8.2f}; error beyond the data {err_out[name]:.3f}{extra}")

In [ ]:
import numpy as np
def test_periodic():
    "for a rhythmic signal, the periodic kernel has the higher optimised marginal likelihood"
    assert best_lml["periodic"] > best_lml["se"] + 3
def test_extrapolate():
    "the periodic kernel extrapolates the rhythm better beyond the data"
    assert err_out["periodic"] < err_out["se"]

test_periodic()
test_extrapolate()
print("All checks passed")